# 歌词采集-QQ音乐

In [1]:
import requests
import re
import json
import os
import time
import pandas as pd
import html
from datetime import datetime
from collections import defaultdict


from collections import Counter

In [2]:
import sys
sys.path.append('..')

# 通用方法

## 时间戳格式化

In [3]:
def format_timestamp(ts, date_format='%Y-%m-%d'):
    """
    自动识别秒或毫秒，并转换为指定格式的字符串
    :param ts: 时间戳 (int 或 float)
    """
    if not ts or ts <= 0:
        return "Unknown"
    
    # 核心逻辑：判断时间戳位数
    # 秒级时间戳目前在 10^9 数量级（10位）
    # 毫秒级时间戳在 10^12 数量级（13位）
    # 我们以 10^11 (11位) 为界限进行区分
    if ts > 100000000000: 
        ts = ts / 1000  # 是毫秒，转换为秒
    
    try:
        dt = datetime.fromtimestamp(ts)
        return dt.strftime(date_format)
    except Exception:
        return "Invalid Date"

## 增量保存到json文件

In [4]:
def save_to_json_list(file_path, song_data):
    """以列表形式保存所有歌曲，避免字典 key 覆盖的问题"""
    data_list = []
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data_list = json.load(f)
                if not isinstance(data_list, list): data_list = []
            except:
                data_list = []

    data_list.append(song_data)

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data_list, f, ensure_ascii=False, indent=4)

# 按歌手采集曲目

In [5]:

def search_song(keyword, page=0):
    """搜索歌曲并返回歌曲ID"""
    url = "https://c.y.qq.com/soso/fcgi-bin/client_search_cp"
    params = {
        "w": keyword,
        "format": "json",
        "n": 50,
        "p": page,  
    }
    headers = {
        "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        data = json.loads(response.text)
        songs = data["data"]["song"]["list"]
        res = []
        for song in songs:
            res_d = {
                "song_id": song["songid"],
                "song_mid": song["songmid"],
                "song_name": song["songname"],
                "song_subname": song["lyric"],
                "artist_name": song["singer"][0]["name"],
                "artist_id": song["singer"][0]["id"],
                "artist_mid": song["singer"][0]["mid"],
                "album_name": song['albumname'],
                "album_id": song['albumid'],
                "album_mid": song['albummid'],
                "duration": song['interval'],
                "publish_time": song["pubtime"],
            }
            res.append(res_d)
        return res
    return []

In [16]:
def get_songs_data_raw(singger, max_page=1):
    """
    获取歌手歌曲列表
    singer_name: 歌手名称
    max_page: 最大页数, 默认每页50条数据，max_page=曲目总数/50
    """
    qq_songs_list = []
    for page in range(0, max_page):
        print(f'正在获取第{page+1}页数据...')
        res = search_song(singger, page)
        time.sleep(2)
        qq_songs_list.extend(res)
    return qq_songs_list

# 曲目过滤

## OST曲目筛选

In [ ]:
# 数据筛选
# 1. artist_name中包含singger
# 2. song_subname中包含书名号，将书名号中的内容保存为新字段: tv_name
def filter_ost_songs(singger, song_list):
    """
    筛选符合条件的歌曲：
    1. 歌手包含 singger
    2. 子标题包含书名号，并提取书名号内容为 tv_name
    """
    filtered_list = []
    # 预编译正则，匹配《 和 》之间最少的内容
    tv_pattern = re.compile(r'《(.*?)》')

    for song in song_list:
        # 条件 1: 校验 artist_name (确保该字段已在之前的解析中生成)
        if singger not in song.get("artist_name", ""):
            continue

        # 条件 2: 校验 song_name，如果”《“在 song_name 中，则跳过
        if "《" in song.get("song_name", ""):
            continue
            
        # 条件 3: 校验 song_subname 并在满足时提取 tv_name
        subname = song.get("song_subname", "")
        match = tv_pattern.search(subname)
        
        if match:
            # 满足条件，创建新字段并保存
            song["tv_name"] = match.group(1)
            filtered_list.append(song)
            
    return filtered_list

## 按专辑列表

In [25]:
def filter_album_songs(singger, song_list, album_list):
    """
    筛选符合条件的歌曲：
    1. 歌手包含 singger
    2. 专辑包含在 album_list 中
    """
    filtered_list = []

    for song in song_list:
        singers = song.get("artist_name", "")
        album_name = song.get("album_name", "")
        if singger in singers and album_name in album_list:
            filtered_list.append(song)
            
    return filtered_list

# 曲目数据清洗

In [ ]:
def clear_song_data(song_data, is_filter_ost=False):
    songs_df = pd.DataFrame(song_data)
    # 分割song_name中的括号
    songs_df = pd.DataFrame(song_data)
    songs_df['song_name_unique'] = songs_df['song_name'].apply(
        lambda x: x.split('(')[0])
    songs_df['song_name_unique'] = songs_df['song_name_unique'].apply(
        lambda x: x.split('（')[0])
    # 删除前后空格
    songs_df['song_name_unique'] = songs_df['song_name_unique'].apply(
        lambda x: x.strip())
    # 按song_name_unique进行去重
    songs_df = songs_df.drop_duplicates(subset=['song_name_unique'],
                                        keep='first')
    # 发行时间格式化
    songs_df['publish_date'] = songs_df['publish_time'].apply(
        lambda x: format_timestamp(x))
    if is_filter_ost:
        # 二次筛选ost， 新建列 is_ost, 如果song_subname中含 影，剧，曲任意一个字，则is_ost为1，否则为0
        songs_df['is_ost'] = songs_df['song_subname'].apply(
            lambda x: 1 if '影' in x or '剧' in x or '曲' in x or '片' in x else 0)
        songs_df = songs_df[songs_df['is_ost'] == 1]
    return songs_df

## 歌词采集

In [66]:
def get_qq_lyric(song_id):
    """根据歌曲ID获取歌词"""
    url = "https://c.y.qq.com/lyric/fcgi-bin/fcg_query_lyric_yqq.fcg"
    params = {
        "nobase64": 1,
        "musicid": song_id,
        "format": "json"
    }
    headers = {
        "Referer": "https://y.qq.com/",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        lyric_data = response.json()
        return lyric_data.get("lyric", "")
    return "歌词获取失败"

In [67]:
def get_all_songs_lyric(file_path, songs_df):
    for i, row in songs_df.iterrows():
        song_id = row['song_id']
        song_name = row['song_name']
        # 判断是否已采集
        if os.path.exists(file_path):
            df = pd.read_json(file_path)
            songs_had = df['song_id'].tolist()
            if song_id not in songs_had:
                print(song_name)
                lyric_raw = get_qq_lyric(song_id)
                single_res = {
                    'song_id': song_id,
                    'song_name': song_name,
                    'lyric_raw': lyric_raw
                }
                save_to_json_list(file_path, single_res)
                time.sleep(2)
        else:
            print(song_name)
            lyric_raw = get_qq_lyric(song_id)
            single_res = {
                    'song_id': song_id,
                    'song_name': song_name,
                    'lyric_raw': lyric_raw
                }
            save_to_json_list(file_path, single_res)
            time.sleep(2)

## 歌词清洗

In [71]:
class QQLyricCleaner:
    def __init__(self):
        self.time_tag_pattern = re.compile(r'\[\d{2,}:\d{2,}\.\d{2,}\]\s*(.*)')
        self.isrc_pattern = re.compile(r'[A-Z]{2}-[A-Z0-9]{3}-\d{2}-\d{5}')
        
        # 扩展黑名单：不仅包含职位，还包含法律/版权声明
        self.exclude_keywords = [
            '词', 'Lyricist', '曲', 'Composer', '编', 'Arranger', 
            '制作', 'Producer', '执行', 'Executive', '合音', 'Chorus',
            '鼓', 'Drums', '钢琴', 'Piano', '录音', 'Engineer', 'Studio',
            '混音', 'Mixing', '母带', 'Mastering', 'OP', 'SP', '版权',
            '：', ':', 'ISRC', '提供', '发行', '编码', 'BY:',
            '统筹', '营销', '推广', '监制', '出品', '弦乐', '唢呐', '和声', '艺人',
            '著作权', '未经', '许可', '翻唱', '翻录', '使用', '不得', '权利人'
        ]

    def _preprocess(self, raw_text):
        if not raw_text: return ""
        # 处理 HTML 实体字符
        text = html.unescape(raw_text)
        # 统一换行与空格
        text = text.replace('&#10;', '\n').replace('\r', '')
        text = text.replace('\u00a0', ' ').replace('&#32;', ' ')
        return text

    def get_credits(self, text):
        """提取制作人信息：作词、作曲、编曲"""
        text = self._preprocess(text)
        credits = {"lyricist": "", "composer": "", "arranger": ""}
        mapping = {
            "lyricist": r"(?:词|作词)(?:\s*[a-zA-Z]+)?\s*[:：]\s*([^\n\r\]]+)",
            "composer": r"(?:曲|作曲)(?:\s*[a-zA-Z]+)?\s*[:：]\s*([^\n\r\]]+)",
            "arranger": r"(?:编曲|Arranger)(?:\s*[a-zA-Z]+)?\s*[:：]\s*([^\n\r\]]+)"
        }
        for key, pattern in mapping.items():
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                # 剔除后面可能跟着的时间戳或括号
                val = re.split(r'[\[\]]', match.group(1).strip())[0].strip()
                credits[key] = val
        return credits

    def get_pure_lyrics(self, raw_text, song_name=None):
        """清洗歌词主体：剔除标题、幕后信息、广告、版权声明"""
        text = self._preprocess(raw_text)
        if not text: return ""

        lines = text.split('\n')
        pure_lyrics = []

        for line in lines:
            match = self.time_tag_pattern.search(line)
            if match:
                content = match.group(1).strip()
                if not content: continue
                
                # --- 核心过滤逻辑 ---
                
                # 1. 过滤特殊括号和推广标签 (如星曜计划)
                if any(tag in content for tag in ['『', '』', '星曜计划']):
                    continue
                
                # 2. 标题行过滤 (命中歌名或包含 歌手+书名号)
                is_title = False
                if song_name and song_name in content: is_title = True
                if '《' in content and '》' in content and ('刘宇宁' in content or '摩登兄弟' in content): is_title = True
                if '-' in content and ('刘宇宁' in content or '摩登兄弟' in content): is_title = True
                if is_title and len(content) < 45: continue

                # 3. 幕后职位及版权声明过滤 (检测行首 15 字)
                content_prefix = content[:15]
                if any(k in content_prefix for k in self.exclude_keywords):
                    continue
                
                # 4. 特殊清洗：如果行内残余“未经著作权人...”这种特定文本，直接跳过
                if re.search(r'未经著作权人|不得翻唱|翻录或使用', content):
                    continue

                if self.isrc_pattern.search(content): continue

                # 5. 格式化：内部空格转逗号
                clean_line = re.sub(r'\s+', ' ', content).replace(" ", "，")
                pure_lyrics.append(clean_line)

        # 返回以句号连接的歌词
        return "。".join(pure_lyrics) + "。" if pure_lyrics else ""

    def process_data(self, raw_lyric, song_id=None, song_name=None):
        """主入口"""
        credits = self.get_credits(raw_lyric)
        lyrics_text = self.get_pure_lyrics(raw_lyric, song_name=song_name)
        
        return {
            "song_id": song_id,
            "song_name": song_name,
            "has_lyric": 1 if lyrics_text else 0,
            "lyricist": credits["lyricist"],
            "composer": credits["composer"],
            "arranger": credits["arranger"],
            "lyrics_text": lyrics_text
        }

In [ ]:
def clear_and_save_lyric(path_prefix, songs_df):
    lyric_raw = pd.read_json(path_prefix+'raw_lyric_data.json')

    cleaner = QQLyricCleaner()
    res_list = []
    for i, row in songs_df.iterrows():
        song_id = row['song_id']
        song_name = row['song_name']
        lyric_raw_single = lyric_raw[lyric_raw['song_id'] == song_id]['lyric_raw'].values[0]
        single_res = cleaner.process_data(lyric_raw_single, song_id, song_name)
        res_list.append(single_res)
    with open(path_prefix+'cleared_lyric_data.json', 'w', encoding='utf-8') as f:
        json.dump(res_list, f, ensure_ascii=False, indent=4)

# main

## 周杰伦

In [19]:
file_path_prefix = "data/jaychou/"
singger = "周杰伦"

### 曲目采集

In [ ]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singger, max_page=20)

正在获取第1页数据...
正在获取第2页数据...
正在获取第3页数据...
正在获取第4页数据...
正在获取第5页数据...
正在获取第6页数据...
正在获取第7页数据...
正在获取第8页数据...
正在获取第9页数据...
正在获取第10页数据...
正在获取第11页数据...
正在获取第12页数据...
正在获取第13页数据...
正在获取第14页数据...
正在获取第15页数据...
正在获取第16页数据...
正在获取第17页数据...
正在获取第18页数据...
正在获取第19页数据...
正在获取第20页数据...


In [10]:
# 原始曲目保存
df_song_data_raw = pd.DataFrame(song_data_raw)
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [63]:
# 曲目筛选，按专辑
album_list = [
    "Jay", "范特西", "八度空间", "叶惠美", "七里香", "十一月的肖邦", "依然范特西", "我很忙", "魔杰座", "跨时代",
    "惊叹号", "十二新作", "哎哟，不错哦", "周杰伦的床边故事", "最伟大的作品"
]
song_data_filted = filter_album_songs(singger, song_data_raw, album_list)

### 数据验证
检查专辑中歌曲是否存在缺失，将缺失歌曲的数据手工添加到song_data_filted中

In [ ]:
song_data_cleared = clear_song_data(song_data_filted)
song_data_cleared

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,97773,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,8220,000MkMni19ClKG,269,1059580800,晴天,2003-07-31
1,102065756,004Z8Ihr0JIu5s,七里香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,七里香,20612,003DFRzD192KKD,299,1091462400,七里香,2004-08-03
2,449205,003aAYrm3GE0Ac,稻香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,36062,002Neh8l0uciQZ,223,1224000000,稻香,2008-10-15
3,410316,002qU5aY3Qu24y,青花瓷,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,我很忙,33021,002eFUFm2XYZ7z,239,1193932800,青花瓷,2007-11-02
4,449198,003cI52o4daJJL,花海,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,36062,002Neh8l0uciQZ,264,1224000000,花海,2008-10-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,3586267,0010jyte3izshw,四季列车,,周杰伦,4558,0025NhlN2yWrP4,十二新作,194021,003Ow85E3pnoqi,159,1356624000,四季列车,2012-12-28
166,107192077,003uYI3j4EDf5q,土耳其冰淇淋,,周杰伦,4558,0025NhlN2yWrP4,周杰伦的床边故事,1458791,003RMaRI1iFoYd,195,1466697600,土耳其冰淇淋,2016-06-24
167,449202,003suDvd4KLUmF,流浪诗人,,周杰伦,4558,0025NhlN2yWrP4,魔杰座,36062,002Neh8l0uciQZ,169,1224000000,流浪诗人,2008-10-15
168,680283,000ES19W2iTudx,嘻哈空姐,,周杰伦,4558,0025NhlN2yWrP4,跨时代,56705,000bviBl4FjTpO,168,1274112000,嘻哈空姐,2010-05-18


In [36]:
song_data_cleared.groupby('album_name')['song_name'].count()

album_name
Jay         10
七里香         10
依然范特西       10
八度空间        10
十二新作        12
叶惠美         11
周杰伦的床边故事     9
惊叹号         11
我很忙         10
最伟大的作品       7
范特西         10
跨时代         11
魔杰座         11
Name: song_name, dtype: int64

In [43]:
song_data_cleared[song_data_cleared['album_name'] == '最伟大的作品']

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
100,361947424,001KhHrT4Mv4pd,红颜如霜,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,28791467,0042cH172YJ0mz,257,1657805400,红颜如霜,2022-07-14
107,361947418,003w2xz20QlUZt,最伟大的作品,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,28791467,0042cH172YJ0mz,244,1657209600,最伟大的作品,2022-07-08
108,361947426,000EqWxe275Yd2,粉色海洋,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,28791467,0042cH172YJ0mz,186,1657805400,粉色海洋,2022-07-14
110,361947419,000Zu3Ah1jb4gl,还在流浪,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,28791467,0042cH172YJ0mz,265,1657805400,还在流浪,2022-07-14
117,361947422,001RtHic1PivYc,错过的烟火,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,28791467,0042cH172YJ0mz,257,1657805400,错过的烟火,2022-07-14
130,361947427,003QrnHn2kR4FD,倒影,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,28791467,0042cH172YJ0mz,234,1657805400,倒影,2022-07-14
145,361947417,000STzer00Hzfu,Intro,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,28791467,0042cH172YJ0mz,29,1657805400,Intro,2022-07-14


In [51]:
songs_list_aa = ['说好不哭', '不爱我就拉倒', 'Mojito', '等你下课', '我是如此相信', '英雄']
for i in song_data_raw:
    if i['song_name'].split(' ')[0] in songs_list_aa:
        print(i)

{'song_id': 212877900, 'song_mid': '001J5QJL1pRQYB', 'song_name': '等你下课 (with 杨瑞代)', 'song_subname': '', 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '等你下课', 'album_id': 3883404, 'album_mid': '003bSL0v4bpKAx', 'duration': 270, 'publish_time': 1516204800}
{'song_id': 237773700, 'song_mid': '001qvvgF38HVc4', 'song_name': '说好不哭 (with 五月天阿信)', 'song_subname': '', 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '说好不哭（with 五月天阿信）', 'album_id': 7876962, 'album_mid': '002gBTVk4JEE2T', 'duration': 222, 'publish_time': 1568646000}
{'song_id': 247261229, 'song_mid': '001PLl3C4gPSCI', 'song_name': '我是如此相信', 'song_subname': '《天火》电影主题曲', 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '我是如此相信', 'album_id': 9612009, 'album_mid': '001hGx1Z0so1YX', 'duration': 266, 'publish_time': 1576339200}
{'song_id': 212877900, 'song_mid': '001J5QJL1pRQYB', 'song_name': '等你下课 (with 杨瑞代)', 'song_su

In [58]:
# 手工添加需要补全的歌曲
songs_to_add = [{'song_id': 105755384, 'song_mid': '004Qscj80GYhGR', 'song_name': '英雄', 'song_subname': '《英雄联盟》中国品牌主题曲', 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '周杰伦的床边故事', 'album_id': 1306793, 'album_mid': '001uJFiE0tbGGa', 'duration': 200, 'publish_time': 1458748800},
 {'song_id': 247261229, 'song_mid': '001PLl3C4gPSCI', 'song_name': '我是如此相信', 'song_subname': '《天火》电影主题曲', 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '最伟大的作品', 'album_id': 9612009, 'album_mid': '001hGx1Z0so1YX', 'duration': 266, 'publish_time': 1576339200},
{'song_id': 268352018, 'song_mid': '001glaI72k8BQX', 'song_name': 'Mojito', 'song_subname': '', 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '最伟大的作品', 'album_id': 12924001, 'album_mid': '0009C3rp3Kfwg0', 'duration': 185, 'publish_time': 1591891200},
{'song_id': 213922043, 'song_mid': '0031TAKo0095np', 'song_name': '不爱我就拉倒', 'song_subname': '', 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '最伟大的作品', 'album_id': 4044657, 'album_mid': '001CnPE31iJ899', 'duration': 245, 'publish_time': 1526313600},
{'song_id': 212877900, 'song_mid': '001J5QJL1pRQYB', 'song_name': '等你下课 (with 杨瑞代)', 'song_subname': '', 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '最伟大的作品', 'album_id': 3883404, 'album_mid': '003bSL0v4bpKAx', 'duration': 270, 'publish_time': 1516204800},
{'song_id': 237773700, 'song_mid': '001qvvgF38HVc4', 'song_name': '说好不哭 (with 五月天阿信)', 'song_subname': '', 'artist_name': '周杰伦', 'artist_id': 4558, 'artist_mid': '0025NhlN2yWrP4', 'album_name': '最伟大的作品', 'album_id': 7876962, 'album_mid': '002gBTVk4JEE2T', 'duration': 222, 'publish_time': 1568646000}
]

### 二次清洗

In [64]:
song_data_filted.extend(songs_to_add)
song_data_cleared = clear_song_data(song_data_filted)
song_data_cleared

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,97773,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,8220,000MkMni19ClKG,269,1059580800,晴天,2003-07-31
1,102065756,004Z8Ihr0JIu5s,七里香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,七里香,20612,003DFRzD192KKD,299,1091462400,七里香,2004-08-03
2,449205,003aAYrm3GE0Ac,稻香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,36062,002Neh8l0uciQZ,223,1224000000,稻香,2008-10-15
3,410316,002qU5aY3Qu24y,青花瓷,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,我很忙,33021,002eFUFm2XYZ7z,239,1193932800,青花瓷,2007-11-02
4,449198,003cI52o4daJJL,花海,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,36062,002Neh8l0uciQZ,264,1224000000,花海,2008-10-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,247261229,001PLl3C4gPSCI,我是如此相信,《天火》电影主题曲,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,9612009,001hGx1Z0so1YX,266,1576339200,我是如此相信,2019-12-15
172,268352018,001glaI72k8BQX,Mojito,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,12924001,0009C3rp3Kfwg0,185,1591891200,Mojito,2020-06-12
173,213922043,0031TAKo0095np,不爱我就拉倒,,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,4044657,001CnPE31iJ899,245,1526313600,不爱我就拉倒,2018-05-15
174,212877900,001J5QJL1pRQYB,等你下课 (with 杨瑞代),,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,3883404,003bSL0v4bpKAx,270,1516204800,等你下课,2018-01-18


In [65]:
song_data_cleared.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

### 歌词采集

In [68]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', song_data_cleared)

晴天
七里香
稻香
青花瓷
花海
搁浅
烟花易冷
兰亭序
说好的幸福呢
明明就
爱你没差
红尘客栈
告白气球
蒲公英的约定
退后
不该 (with aMEI)
爱在西元前
最长的电影
说了再见
园游会
我落泪情绪零碎
半岛铁盒
东风破
我不配
雨下一整晚
暗号
听妈妈的话
安静
给我一首歌的时间
你听得到
反方向的钟
借口
简单爱
回到过去
爱情废柴
以父之名
开不了口
彩虹
大笨钟
夜的第七章
龙卷风
爱的飞行日记
外婆
千里之外
阳光宅男
分裂
可爱女人
Mine Mine
超人不会飞
爷爷泡的茶
星晴
黑色幽默
白色风车
心雨
好久不见
止战之殇
菊花台
半兽人
双截棍
她的睫毛
本草纲目
对不起
红颜如霜
米兰的小铁匠
甜甜的
上海一九四三
一点点
牛仔很忙
最后的战役
最伟大的作品
粉色海洋
哪里都是你
还在流浪
龙拳
威廉古堡
乱舞春秋
我的地盘
自导自演
乌克丽丽
错过的烟火
火车叨位去
时光机
将军
手语
红模仿
说走就走
完美主义
伊斯坦堡
公主病
你好吗
爱情悬崖
忍者
倒影
三年二班
迷迭香
印第安老斑鸠
超跑女神
梯田
傻笑
龙战骑士
床边故事
娘子
爸，我回来了
前世情人
双刀
世界未末日
乔克叔叔
Intro
跨时代
同一种调调
斗牛
琴伤
困兽之斗
疗伤烧肉粽
公公偏头痛
水手怕水
无双
Now You See Me
梦想启动
懦夫
扯
迷魂曲
皮影戏
蛇舞
魔术先生
惊叹号
免费教学录影带
四季列车
土耳其冰淇淋
流浪诗人
嘻哈空姐
比较大的大提琴
英雄
我是如此相信
Mojito
不爱我就拉倒
等你下课 (with 杨瑞代)
说好不哭 (with 五月天阿信)


### 歌词清洗

In [72]:
clear_and_save_lyric(file_path_prefix, song_data_cleared)